In [20]:
import os
import sys
import glob
import random
import re
import collections
import time
import numpy as np
import pandas as pd
import pydicom
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch.utils import data as torch_data
from sklearn import model_selection as sk_model_selection
from torch.nn import functional as torch_functional
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [58]:
mri_types = ['FLAIR', 'T1w', 'T1wCE', 'T2W']
SIZE = 256
NUM_IMAGES = 64
BATCH_SIZE = 4
N_EPOCHS = 16
SEED = 12345
LEARNING_RATE = 0.0005
LR_DECAY = 0.9
data_directory = "dataset/rsna-miccai-brain-tumor-radiogenomic-classification/"

In [63]:
def load_dicom_image(path, img_size=SIZE):
    dicom = pydicom.dcmread(path) # read_file --> dcmread
    data = dicom.pixel_array
    if np.min(data)==np.max(data):
        data = np.zeros((img_size,img_size))
        return data
    
    data = cv2.resize(data, (img_size, img_size))
    return data


def natural_sort(l): 
    convert = lambda text: int(text) if text.isdigit() else text.lower()
    alphanum_key = lambda key: [convert(c) for c in re.split('([0-9]+)', key)]
    return sorted(l, key=alphanum_key)


def load_dicom_images_3d(scan_id, num_imgs=NUM_IMAGES, img_size=SIZE, mri_type="FLAIR", split="train"):
    files = natural_sort(glob.glob(f"{data_directory}/{split}/{scan_id}/{mri_type}/*.dcm"))
    every_nth = len(files) / num_imgs
    indexes = [min(int(round(i*every_nth)), len(files)-1) for i in range(0,num_imgs)]
    
    files_to_load = [files[i] for i in indexes]
    
    img3d = np.stack([load_dicom_image(f) for f in files_to_load]).T 
    
    img3d = img3d - np.min(img3d)
    if np.max(img3d) != 0:
        img3d = img3d / np.max(img3d)
    
    return np.expand_dims(img3d,0)


load_dicom_images_3d("00000", mri_type=mri_types[0]).shape

(1, 256, 256, 64)

In [69]:
def set_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed) # torch의 CPU 시드 결정
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed) # torch의 GPU 시드 결정
        torch.backends.cudnn.deterministic=True # cudnn에서 가능한 결정적 알고리즘만 사용하도록 강제.

set_seed(SEED)

In [75]:
samples_to_exclude = [109,123,709] # 제외할 샘플, exclude

train_df = pd.read_csv(f'{data_directory}/train_labels.csv')
train_df.info()
print('original-shape', train_df.shape)

train_df = train_df[~train_df.BraTS21ID.isin(samples_to_exclude)] # BraTS21ID 컬럼명
print("new-shape", train_df.shape)

df_train, df_valid = sk_model_selection.train_test_split(
    train_df,
    test_size=0.2,
    random_state=SEED,
    stratify=train_df["MGMT_value"],
)

<class 'pandas.DataFrame'>
RangeIndex: 585 entries, 0 to 584
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   BraTS21ID   585 non-null    int64
 1   MGMT_value  585 non-null    int64
dtypes: int64(2)
memory usage: 9.3 KB
original-shape (585, 2)
new-shape (582, 2)


In [77]:
df_train.tail()

,BraTS21ID,MGMT_value
63,99,0
130,195,0
82,124,0
177,262,0
557,811,1


In [81]:
class Dataset(torch_data.Dataset):
    def __init__(self, paths, targets=None, mri_type=None, split='train'):
        self.paths = paths
        self. targets = targets
        self.mri_type = mri_type
        self.split = split

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index): # 인스턴스를 리스트처럼 인덱스로 접근하게 만들어줌.
        scan_id = self.paths[index]
        if self.targets is None:
            data = load_dicom_images_3d(str(scan_id).zfill(5), mri_type=self.mri_type, split=self.split)
        else:
            data = load_dicom_images_3d(str(scan_id).zfill(5), mri_type=self.mri_type, split='train')

        if self.targets is None:
            return {'X' : data, 'id':scan_id}
        else:
            return {'X' : data, 'y':torch.tensor(self.targets[index], dtype=torch.float)}
            

In [87]:
from monai.networks.nets.densenet import DenseNet121

def build_model():
    model = DenseNet121(spatial_dims=3, in_channels=1, out_channels=1)
    return model

In [ ]:
class Trainer:
    def __init__(
        self,
        model,
        device,
        optimizer,
        criterion
    ):
    """
    criterion : loss 함수의 객체
    n_patience : early stopping 을 위한 카운트
    ====
    AUC --> classification 분류 모델의 성능 지표
    """
        self.model = model
        self.device = device
        self.optimizer = optimizer
        self.criterion = criterion
        self.best_valid_score =.0
        self.n_patience = 0
        self.lastmodel = None

        self.val_losses = []
        self.train_losses = []
        self.val_auc = []

    def fit(self, epochs, train_loader, valid_loader, save_path, patience):
        """
        patience : early stopping 의 기준.
        """
        for n_epoch in range(1, epoch + 1):
            self.info_message("EPOCH:{}", n_epoch)

            train_loss, train_time = self.train_epoch(train_loader)
            valid_loss, valid_auc, valid_time = self.valid_epoch(valid_loader)

            self.train_losses.append(train_loss)
            self.val_losses.append(valid_loss)
            self.val_auc.append(valid_auc)

            self.info_message(
                "[Epoch Train: {}] loss: {:.4f}, time: {:.2f} s",
                n_epoch, train_loss, train_time
            )
            
            self.info_message(
                "[Epoch Valid: {}] loss: {:.4f}, auc: {:.4f}, time: {:.2f} s",
                n_epoch, valid_loss, valid_auc, valid_time
            )

            if self.best_valid_score < valid_auc: 
                self.save_model(n_epoch, save_path, valid_loss, valid_auc)
                self.info_message(
                     "auc improved from {:.4f} to {:.4f}. Saved model to '{}'", 
                    self.best_valid_score, valid_auc, self.lastmodel
                )
                self.best_valid_score = valid_auc
                self.n_patience = 0
            else:
                self.n_patience += 1

            if self.n_patience >= patience:
                self.info_message("\nValid auc didn't improve last {} epochs.", patience)
                break